# Lab 2, Notebook 1 — Normalized TF and IDF Variants

Reuses the exact dataset built in Lab 1 (`Lab1/data/{train,val,test}.csv` — same 23 languages,
same 800/100/100 stratified split, same sentences) and the same combined word-1/2-gram +
char-2/3/4-gram tokenizer. Nothing about the data changes; what changes is how term weights are
computed.

Lab 1 used **unnormalized TF** (raw counts) and **unnormalized IDF**
(`ln((1+N)/(1+df(t))) + 1`). This notebook adds:

1. **TF normalized by sentence length** — `TF(t,d) = count(t,d) / (number of words in d)`
2. **TF normalized by the most frequent word in the sentence** — `TF(t,d) = count(t,d) / (count of the most frequent word in d)`
3. **IDF normalized by the most frequent word across the corpus** — replaces the `ln((1+N)/(1+df))+1`
   formula with `ln(M / df(t))`, where `M` is the document frequency of the single most frequent
   *word* (not char n-gram) in the training vocabulary.

Combined with the existing unnormalized TF and IDF, that's **3 TF variants × 2 IDF variants =
6 combinations**, all built here and saved for Notebook 2 to train and compare.


## Design choices, stated explicitly

- **"Words" means word-unigram tokens** (`text.split()`), for both new normalizations — even
  though the normalizer is then applied uniformly to *every* feature in a document's vector,
  including character n-grams. This is the literal reading of the assignment text ("total number
  of words," "most frequent word"), not an approximation.
- **One consequence worth flagging honestly:** because `M` (for IDF normalization) is computed
  only from word-unigram document frequencies, and character n-grams are often more universally
  common than any single word (a word belongs to one language's vocabulary; a character bigram
  like a space + a common letter can occur across every language sharing that script), it is
  possible for a character n-gram's `df(t)` to exceed `M`. When that happens, `ln(M/df(t))` is
  **negative** — the term actively subtracts from a document's score under this scheme. This was
  checked empirically below rather than assumed away: it affects a small, specific slice of the
  vocabulary (extremely common script-level character bigrams), not a design flaw.
- **No final L2 row-normalization is applied in this lab** (Lab 1 did apply one). The reasoning
  is worked through in detail further down, once it is empirically motivated — in short,
  L2-normalizing after a per-document-constant TF rescaling always cancels that rescaling, so
  keeping it here would make 3 of the 6 combinations secretly identical to the other 3.
- **The vocabulary itself is fit once, shared across all 6 combinations** (same `min_df=3`
  pruning as Lab 1). This isolates the comparison to the weighting formula alone — all 6
  combinations see exactly the same set of features, differing only in how those features are
  weighted.


In [1]:
import json
import os
import time
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp

LAB1_DATA = "../../Lab1/data"
DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)

train_df = pd.read_csv(f"{LAB1_DATA}/train.csv")
val_df = pd.read_csv(f"{LAB1_DATA}/val.csv")
test_df = pd.read_csv(f"{LAB1_DATA}/test.csv")
print(train_df.shape, val_df.shape, test_df.shape)

for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    df[["label"]].to_csv(f"{DATA_DIR}/y_{split_name}.csv", index=False)


(18400, 2) (2300, 2) (2300, 2)


In [2]:
def tokenize(text):
    """Identical to Lab 1: word 1/2-grams + char 2/3/4-grams, tagged by type."""
    text = text.strip()
    words = text.split()
    grams = []
    for w in words:
        grams.append("W1|" + w)
    for i in range(len(words) - 1):
        grams.append("W2|" + words[i] + " " + words[i + 1])
    for n in (2, 3, 4):
        for i in range(len(text) - n + 1):
            grams.append(f"C{n}|" + text[i:i + n])
    return grams


## Fit the shared vocabulary once

Same `min_df=3` pruning as Lab 1. While counting document frequency, also separately track
document frequency restricted to word-unigram (`W1|`) grams, needed for the IDF-maxdf formula.


In [3]:
MIN_DF = 3

t0 = time.time()
df_counter = Counter()
word_df_counter = Counter()
for text in train_df["text"]:
    grams = set(tokenize(text))
    df_counter.update(grams)
    word_df_counter.update(g for g in grams if g.startswith("W1|"))
t1 = time.time()

vocab_terms = sorted(g for g, c in df_counter.items() if c >= MIN_DF)
vocab = {g: i for i, g in enumerate(vocab_terms)}
N = len(train_df)

print(f"vocab fit in {t1 - t0:.1f}s -> vocab size = {len(vocab):,}")

M = max(word_df_counter.values())
best_word = max(word_df_counter, key=word_df_counter.get)
print(f"most frequent word by document frequency: {best_word!r}  df={M}  (out of N={N} training docs)")


vocab fit in 1.8s -> vocab size = 214,895
most frequent word by document frequency: 'W1|।'  df=1415  (out of N=18400 training docs)


In [4]:
df_arr = np.array([df_counter[g] for g in vocab_terms], dtype=np.float64)

idf_standard = np.log((1.0 + N) / (1.0 + df_arr)) + 1.0
idf_maxdf = np.log(M / df_arr)

n_negative = int((idf_maxdf < 0).sum())
print(f"idf_standard: min={idf_standard.min():.3f} max={idf_standard.max():.3f} mean={idf_standard.mean():.3f}")
print(f"idf_maxdf:    min={idf_maxdf.min():.3f} max={idf_maxdf.max():.3f} mean={idf_maxdf.mean():.3f}")
print(f"vocab terms with negative idf_maxdf (df(t) > M): {n_negative} / {len(vocab_terms)} "
      f"({n_negative / len(vocab_terms):.4%})")

neg_examples = [vocab_terms[i] for i in np.where(idf_maxdf < 0)[0][:8]]
print("examples:", neg_examples)


idf_standard: min=2.462 max=9.434 mean=8.625
idf_maxdf:    min=-1.103 max=6.156 mean=5.222
vocab terms with negative idf_maxdf (df(t) > M): 59 / 214895 (0.0275%)
examples: ['C2| ا', 'C2| م', 'C2| ک', 'C2| अ', 'C2| आ', 'C2| क', 'C2| ग', 'C2| ज']


## Tokenize each split exactly once, reuse across all 6 weighting combinations

Only the weighting formula differs between the 6 combinations — the tokenization and raw counts
are identical every time. Computing them once per split and reusing them for all 6 combinations
avoids six redundant tokenization passes.


In [5]:
def precompute_counts(texts):
    """For each document: raw {vocab_index: count} for in-vocab grams, plus the two
    document-level denominators needed for tf_mode="len" and tf_mode="max"."""
    per_doc_counts = []
    n_words_list = []
    max_word_freq_list = []

    for text in texts:
        grams = tokenize(text)
        raw_counts = Counter(grams)
        indexed = {}
        for gram, count in raw_counts.items():
            j = vocab.get(gram)
            if j is not None:
                indexed[j] = count
        per_doc_counts.append(indexed)

        words = text.strip().split()
        n_words_list.append(float(len(words)) or 1.0)
        word_counts = Counter(words)
        max_word_freq_list.append(float(max(word_counts.values())) if word_counts else 1.0)

    return per_doc_counts, np.array(n_words_list), np.array(max_word_freq_list)


t0 = time.time()
train_counts, train_nwords, train_maxword = precompute_counts(train_df["text"].tolist())
val_counts, val_nwords, val_maxword = precompute_counts(val_df["text"].tolist())
test_counts, test_nwords, test_maxword = precompute_counts(test_df["text"].tolist())
t1 = time.time()
print(f"tokenized all 3 splits in {t1 - t0:.1f}s")


tokenized all 3 splits in 2.3s


## A mathematical catch, found empirically before it wasted the whole comparison

The first version of `build_matrix` below L2-normalized every row, matching Lab 1. Building all
6 combinations that way and comparing them revealed that `raw_std`, `len_std`, and `max_std`
were **numerically identical** (element-wise differences at floating-point noise level, ~1e-16) —
and likewise for the three `_maxdf` variants. The reason is a general fact, not a bug: `len` and
`max` TF normalization both divide every term in a document's vector by one constant scalar
specific to that document. Any standard vector norm is homogeneous of degree 1
(`‖c·v‖ = c·‖v‖` for a positive scalar `c`), so `v/‖v‖ = (c·v)/‖c·v‖` — **L2-normalizing the
final vector always cancels a prior per-document constant rescaling, no matter what that
constant was.** Dividing by sentence length or by the max word count, then L2-normalizing,
produces exactly the same vector as never dividing at all.

That means keeping the L2 step would silently collapse 6 combinations into 2, defeating the
entire comparison this lab asks for. So the final vector-normalization step used in Lab 1 is
**not applied here** — the 3 TF variants are left to actually differ in the vectors the
classifier sees, which is the only way to honestly test what this lab is asking.


In [6]:
def build_matrix(per_doc_counts, denom_array, idf_array, tf_mode):
    rows, cols, data = [], [], []
    for row_idx, indexed in enumerate(per_doc_counts):
        if not indexed:
            continue
        denom = denom_array[row_idx] if tf_mode != "raw" else 1.0
        for j, count in indexed.items():
            tf = count / denom
            rows.append(row_idx)
            cols.append(j)
            data.append(tf * idf_array[j])
    return sp.csr_matrix((data, (rows, cols)), shape=(len(per_doc_counts), len(vocab)))


TF_MODES = {
    "raw": None,           # denom handled as constant 1.0 inside build_matrix
    "len": "n_words",
    "max": "max_word_freq",
}
IDF_MODES = {
    "std": idf_standard,
    "maxdf": idf_maxdf,
}

SPLITS = {
    "train": (train_counts, train_nwords, train_maxword),
    "val": (val_counts, val_nwords, val_maxword),
    "test": (test_counts, test_nwords, test_maxword),
}


In [7]:
results_summary = []
t_start = time.time()

for tf_mode in ("raw", "len", "max"):
    for idf_name, idf_array in IDF_MODES.items():
        combo = f"{tf_mode}_{idf_name}"
        for split_name, (counts, nwords, maxword) in SPLITS.items():
            denom_array = nwords if tf_mode == "len" else (maxword if tf_mode == "max" else None)
            t0 = time.time()
            X = build_matrix(counts, denom_array, idf_array, tf_mode)
            dt = time.time() - t0
            sp.save_npz(f"{DATA_DIR}/X_{split_name}_{combo}.npz", X)
            if split_name == "train":
                results_summary.append({
                    "combo": combo, "shape": X.shape, "nnz": X.nnz, "seconds": round(dt, 2),
                })

print(f"built and saved 18 matrices (6 combinations x 3 splits) in {time.time() - t_start:.1f}s total\n")
pd.DataFrame(results_summary)


built and saved 18 matrices (6 combinations x 3 splits) in 20.0s total



,combo,shape,nnz,seconds
0,raw_std,"(18400, 214895)",3690535,1.34
1,raw_maxdf,"(18400, 214895)",3690535,1.30
2,len_std,"(18400, 214895)",3690535,1.39
3,len_maxdf,"(18400, 214895)",3690535,1.40
4,max_std,"(18400, 214895)",3690535,1.42
5,max_maxdf,"(18400, 214895)",3690535,1.39


## Sanity check: same vocabulary size across all 6, but different weight distributions


In [8]:
for tf_mode in ("raw", "len", "max"):
    for idf_name in ("std", "maxdf"):
        combo = f"{tf_mode}_{idf_name}"
        X = sp.load_npz(f"{DATA_DIR}/X_train_{combo}.npz")
        nz = X.data
        print(f"{combo:10s} shape={X.shape}  nnz={X.nnz:8d}  "
              f"weight[min={nz.min():.4f} mean={nz.mean():.4f} max={nz.max():.4f}]")


raw_std    shape=(18400, 214895)  nnz= 3690535  weight[min=2.4622 mean=7.6657 max=684.9093]


raw_maxdf  shape=(18400, 214895)  nnz= 3690535  weight[min=-23.2772 mean=3.5355 max=279.3985]


len_std    shape=(18400, 214895)  nnz= 3690535  weight[min=0.0425 mean=0.5622 max=96.1276]


len_maxdf  shape=(18400, 214895)  nnz= 3690535  weight[min=-1.4109 mean=0.2606 max=39.2138]


max_std    shape=(18400, 214895)  nnz= 3690535  weight[min=0.2127 mean=5.5679 max=486.6461]


max_maxdf  shape=(18400, 214895)  nnz= 3690535  weight[min=-23.2772 mean=2.5776 max=198.5200]
